# OpenPlaque RCA Centerline Validation — Run All
Choose **Runtime → Run all**. The notebook will install OpenPlaque, ask you to upload the CCTA and mask, automatically identify which uploaded file is the CT versus mask, derive a provisional RCA component and automatic endpoint seeds, extract the centerline, and display 0/10/50 mm landmarks. No path or coordinate editing is required.

Research prototype only. Visually verify the RCA path before using it for quantitative PCAT analysis.

In [ ]:
!pip -q install git+https://github.com/pazzani/OpenPlaque.git@rca-centerline-prototype
import os, numpy as np, SimpleITK as sitk, matplotlib.pyplot as plt
from scipy import ndimage as ndi
from google.colab import files
from openplaque.centerline import extract_rca_centerline, show_centerline_mip
print('OpenPlaque loaded.')


## Upload the two NIfTI files
Upload the original CCTA and its OpenPlaque/nnU-Net coronary segmentation (`.nii` or `.nii.gz`). Run All pauses here for the upload dialog and then continues automatically.

In [ ]:
uploaded = files.upload()
paths = ['/content/' + name for name in uploaded]
nifti = [p for p in paths if p.lower().endswith(('.nii','.nii.gz'))]
if len(nifti) != 2:
    raise ValueError(f'Please upload exactly two NIfTI files (CCTA + segmentation); received {len(nifti)}: {nifti}')
imgs = [sitk.ReadImage(p) for p in nifti]
arrs = [sitk.GetArrayFromImage(im) for im in imgs]
if arrs[0].shape != arrs[1].shape:
    raise ValueError(f'Uploaded image shapes differ: {arrs[0].shape} vs {arrs[1].shape}')
# CT normally has many HU values; segmentation has few discrete values.
nu = [len(np.unique(a)) if a.size < 5_000_000 else len(np.unique(a.ravel()[::max(1,a.size//1_000_000)])) for a in arrs]
mask_i = int(np.argmin(nu)); ct_i = 1-mask_i
ct_img, mask_img = imgs[ct_i], imgs[mask_i]
ct, mask_raw = arrs[ct_i], arrs[mask_i]
CT_PATH, MASK_PATH = nifti[ct_i], nifti[mask_i]
spacing_xyz = ct_img.GetSpacing()
print('CT:', os.path.basename(CT_PATH), 'shape', ct.shape, 'spacing xyz', spacing_xyz)
print('Mask:', os.path.basename(MASK_PATH), 'labels', np.unique(mask_raw))


## Automatically form a provisional RCA candidate
The notebook treats all nonzero coronary labels as vessel foreground, finds connected components, and selects a long vessel component using a right-sided anatomical heuristic. This is intentionally provisional: the resulting overlays are the validation step.

In [ ]:
foreground = mask_raw > 0
labels, nlab = ndi.label(foreground, structure=ndi.generate_binary_structure(3,3))
components=[]
for lab in range(1,nlab+1):
    pts=np.argwhere(labels==lab); n=len(pts)
    if n < 20: continue
    span=np.ptp(pts,axis=0); centroid=pts.mean(axis=0)
    components.append((lab,n,span,centroid))
if not components: raise ValueError('No usable coronary component found in segmentation.')
# Prefer substantial elongated components; among comparable ones favor larger x centroid.
components.sort(key=lambda q:(q[1], q[2].max()), reverse=True)
top=components[:min(6,len(components))]
size_cut=0.20*top[0][1]
eligible=[q for q in top if q[1]>=size_cut]
chosen=max(eligible,key=lambda q:(q[3][2], q[2].max()))
lab,nvox,span,centroid=chosen
rca = labels==lab
pts=np.argwhere(rca)
# Automatic endpoints: farthest pair approximation in physical coordinates.
sp_zyx=np.asarray(spacing_xyz)[::-1]
p0=pts[np.argmin(pts[:,2])]
d=np.linalg.norm((pts-p0)*sp_zyx,axis=1); p1=pts[np.argmax(d)]
d=np.linalg.norm((pts-p1)*sp_zyx,axis=1); p2=pts[np.argmax(d)]
# Provisional ostium: endpoint with larger x coordinate (right-sided RCA heuristic).
if p2[2] > p1[2]: p1,p2=p2,p1
OSTIUM_ZYX=tuple(int(v) for v in p1); DISTAL_HINT_ZYX=tuple(int(v) for v in p2)
print('Connected components:', nlab)
print('Chosen RCA candidate voxels:', int(rca.sum()), 'span zyx:', span)
print('Automatic ostium zyx:', OSTIUM_ZYX)
print('Automatic distal hint zyx:', DISTAL_HINT_ZYX)


In [ ]:
result = extract_rca_centerline(rca, spacing_xyz, OSTIUM_ZYX, distal_hint_zyx=DISTAL_HINT_ZYX, landmark_distances_mm=(0.,10.,50.))
print(f'Centerline length: {result.length_mm:.1f} mm')
print('Snapped ostium:', result.ostium_zyx_voxel, ' endpoint:', result.endpoint_zyx_voxel)
print('Landmarks:', sorted(result.landmarks_xyz_mm))
if 50. not in result.landmarks_xyz_mm: print('WARNING: path is shorter than 50 mm.')


## Visual validation
The path should follow the RCA continuously without jumping branches. Confirm especially that 10 mm and 50 mm lie on the same main RCA path.

In [ ]:
for axis,name in [(0,'axial MIP'),(1,'coronal MIP'),(2,'sagittal MIP')]:
    fig,ax=show_centerline_mip(ct,result,axis=axis)
    ax.set_title(f'{name}: provisional RCA centerline ({result.length_mm:.1f} mm)')
    plt.show()
ptsmm=result.points_xyz_mm
seg=np.linalg.norm(np.diff(ptsmm,axis=0),axis=1) if len(ptsmm)>1 else np.array([])
direct=np.linalg.norm(ptsmm[-1]-ptsmm[0]) if len(ptsmm)>1 else 0.
print('Centerline voxels:',len(ptsmm))
print('Median/max step mm:', (float(np.median(seg)),float(np.max(seg))) if len(seg) else 'n/a')
print('Path/direct ratio:', round(result.length_mm/direct,3) if direct else 'n/a')
print('\nPASS only if: correct RCA; no branch jump; path near lumen center; 10 and 50 mm landmarks anatomically plausible.')
